# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Dataset Exploration with `mlcroissant`
This notebook provides a guided exploration of the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Install `mlcroissant` if missing
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load dataset metadata and object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset name:", metadata.name)
print("Description:", metadata.description)
print("Version:", getattr(metadata, 'version', 'N/A'))
print("Authors (@id):", [a['@id'] for a in getattr(metadata, 'author', [])])

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset contains several tabular record sets, each one defined in the Croissant schema by an `@id`. We will list all available record sets and their field `@id`s. This helps us reference and load data precisely.

In [ ]:
# List all record sets and their fields by @id

record_sets = dataset.record_sets

overview = {}
for rs in record_sets:
    print(f"Record Set: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    field_ids = [f['@id'] if isinstance(f, dict) else f for f in fields]
    print("  Fields (@id):", field_ids)
    overview[rs['@id']] = field_ids
    print()

# Example: print first record from each record set
for rs in record_sets:
    rs_id = rs['@id']
    print(f"Sample record from {rs_id}:")
    # Print first record
    try:
        recs = dataset.records(record_set=rs_id)
        rec = next(recs)
        print(rec)
    except Exception as e:
        print("Could not load record:", e)
    print()

## 3. Data Extraction
Load data from a specific record set into Pandas DataFrames for analysis. Use the record set and field `@id`s identified above.

We will load all available record sets and display the columns and the first rows.

In [ ]:
# Extract all record sets into Pandas DataFrames

# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"\nRecordSet: {rs_id}")
        print("Columns (@id):", df.columns.tolist())
        print(df.head(), "\n")
    except Exception as e:
        print(f"Could not load RecordSet {rs_id}: {e}\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, and grouping data. All fields are referenced by their `@id`.

For demonstration, we select a numeric field from the first record set and apply filtering and normalization.

In [ ]:
# Select record set and numeric field for analysis

# Choose first available record set and field, adapt as needed
if len(record_set_ids) > 0:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    columns = df.columns.tolist()
    print(f"Columns (@id) in {record_set_id}: {columns}")

    # Identify a numeric field by checking dtypes
    numeric_fields = [col for col in columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_fields:
        # Try to coerce some likely candidates
        for col in columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            except:
                pass
        numeric_fields = [col for col in columns if pd.api.types.is_numeric_dtype(df[col])]

    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field (@id): {numeric_field_id}")

        # Filter: records where value > threshold
        threshold = df[numeric_field_id].quantile(0.5)
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by first non-numeric field (if any)
        group_field = None
        for col in columns:
            if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
                group_field = col
                break
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found in the selected record set. Skipping EDA.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize numeric data distributions or relationships between fields in the dataset.

We plot a histogram of the chosen numeric field, and if grouping was applied, a barplot of group means.

In [ ]:
# Visualization for numeric data
if len(record_set_ids) > 0 and numeric_fields:
    plt.figure(figsize=(6, 4))
    filtered_df[numeric_field_id].hist(bins=10, alpha=0.7)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouped_df exists, plot group means
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(6, 4))
        plt.bar(grouped_df[group_field].astype(str), grouped_df[numeric_field_id])
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR^2 dataset using the `mlcroissant` library, referencing all record sets and fields by their `@id`. We've extracted data, filtered and normalized a numeric field, grouped by categorical variables, and visualized the results. Due to the dataset's small size and structure (N=3, clinical records), further statistical or ML modeling must be performed with caution.

For in-depth analysis, consult the original Croissant schema fields and refer to each field and entity precisely by their `@id`.